# 1b · Does percentile sensitivity vary by development context?

**Research question (RQ 1b):** *Does IQB parameter sensitivity vary by development context, particularly in countries with uneven infrastructure that likely exhibit wider measurement distributions? Their IQB scores and rankings may shift more dramatically across percentile thresholds than countries with uniformly developed networks.*

Which thresholds impact the score the most? 

In [ ]:
unsure = R[(R.display_2025 == "show a range")].sort_values("n_2025", ascending=False)
print(f"{len(unsure)} countries need a range in 2025. The pass rate closest to 50 % decides it:")
print(unsure.closest_test_p50.value_counts().to_string())
unsure[["country", "income_group", "n_2025", "iqb_p50", "range_p50_lo", "range_p50_hi", "conf_p50_2025",
        "closest_test_p50", "closest_pass_rate"]].head(12).round(3)

## Kendall's τ by income group and by ISP concentration

τ between the rankings produced by two percentiles, computed **within** each group (τ-b, which handles ties), with a 95 % bootstrap interval from resampling countries within the group.

In [ ]:
PAIRS = [(25, 50), (50, 75), (75, 95), (50, 95), (25, 75)]


def tau_ci(a, b, n_boot=1000):
    a, b = np.asarray(a), np.asarray(b)
    t = kendalltau(a, b).statistic
    bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(a), len(a))
        bs.append(kendalltau(a[i], b[i]).statistic)
    lo, hi = np.nanpercentile(bs, [2.5, 97.5])
    return t, lo, hi


def tau_table(groups):
    out = []
    for name, idx in groups.items():
        for p, q in PAIRS:
            t, lo, hi = tau_ci(S.loc[idx, p], S.loc[idx, q])
            out.append((name, f"P{p} vs P{q}", t, lo, hi, len(idx)))
    return pd.DataFrame(out, columns=["group", "pair", "tau", "lo", "hi", "n"])


groups_inc = {
    "All": d.index,
    **{T.INCOME_SHORT[i]: d.index[d.income == i] for i in T.INCOME_ORDER},
}
groups_hhi = {
    h: d.index[d.hhi_tercile == h] for h in ["Low HHI", "Mid HHI", "High HHI"]
}
tau_inc, tau_hhi = tau_table(groups_inc), tau_table(groups_hhi)
tau_all = pd.concat([tau_inc, tau_hhi])
tau_all.pivot(index="group", columns="pair", values="tau").loc[
    list(groups_inc) + list(groups_hhi), [f"P{p} vs P{q}" for p, q in PAIRS]
].round(2)
